<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing

In [25]:
!pip install gradio matplotlib -q

In [26]:
import gradio as gr
import pandas as pd
import gspread
import datetime
import matplotlib.pyplot as plt
from google.colab import auth
from google.auth import default

In [27]:
# 安裝中文字體
!sudo apt-get install -y fonts-noto-cjk

# 清除 matplotlib 字體快取
!rm -rf ~/.cache/matplotlib

# 配置 matplotlib 使用中文字體
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP'] # 使用 Noto Sans CJK JP 字體
plt.rcParams['axes.unicode_minus'] = False # 解決負號顯示問題

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-noto-cjk is already the newest version (1:20220127+repack1-1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [28]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_URL = 'https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing'
gsheets = gc.open_by_url(SHEET_URL)
worksheet = gsheets.worksheet('工作表1')

In [30]:
# 3. 功能函式定義(font issue can't be solve right now maybe I should switch to English)

def get_raw_data():
    """讀取試算表原始資料並回傳 DataFrame"""
    data = worksheet.get_all_values()
    if len(data) <= 1:
        return pd.DataFrame(columns=["日期", "時間", "分類", "品項", "金額", "地點", "支付方式"])
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

def get_pie_chart():
    """讀取試算表資料並生成圓餅圖"""
    data = worksheet.get_all_values()
    if len(data) <= 1:
        return None

    df = pd.DataFrame(data[1:], columns=data[0])
    df['金額'] = pd.to_numeric(df['金額'], errors='coerce').fillna(0)
    category_sums = df.groupby('分類')['金額'].sum()

    fig, ax = plt.subplots(figsize=(6, 6))
    category_sums.plot(kind='pie', autopct='%1.1f%%', ax=ax, startangle=140)
    ax.set_ylabel('')
    ax.set_title('支出分類佔比')
    return fig

def submit_to_sheet(date_str, time_str, category, item, amount, location, payment_method):
    """寫入資料並返回狀態與更新後的圖表及表格"""
    try:
        datetime.datetime.strptime(date_str, '%Y-%m-%d')
        new_row = [[date_str, time_str, category, item, str(amount), location, payment_method]]
        worksheet.append_rows(values=new_row, value_input_option='USER_ENTERED')

        new_plot = get_pie_chart()
        new_df = get_raw_data()
        return f"成功寫入！{item} (${amount})", new_plot, new_df
    except Exception as e:
        return f"發生錯誤：{str(e)}", None, None

# 4. Gradio 介面配置
with gr.Blocks() as demo:
    gr.Markdown("# 日常支出速算與視覺化")

    with gr.Tab("支出紀錄"):
        with gr.Row():
            with gr.Column():
                date_in = gr.Textbox(label="日期 (YYYY-MM-DD)", value=datetime.date.today().isoformat())
                time_in = gr.Textbox(label="時間", value=datetime.datetime.now().strftime("%H:%M"))
                cat_in = gr.Dropdown(choices=["食物", "飲料", "交通", "書", "其他"], label="支出類別", value="食物")
                item_in = gr.Textbox(label="品項")
                amt_in = gr.Number(label="金額")
                location_in = gr.Textbox(label="地點")
                payment_in = gr.Dropdown(choices=["現金", "信用卡", "電子支付", "轉帳"], label="支付方式", value="現金")
                btn = gr.Button("記錄支出", variant="primary")
                status = gr.Textbox(label="狀態")

            with gr.Column():
                plot_output = gr.Plot(label="支出分析圖")
                refresh_chart_btn = gr.Button("重新整理圓餅圖")

    with gr.Tab("檢視原始資料"):
        data_table = gr.DataFrame(label="Google Sheet 原始資料")
        refresh_table_btn = gr.Button("重新整理資料表")

    # 事件綁定
    btn.click(
        fn=submit_to_sheet,
        inputs=[date_in, time_in, cat_in, item_in, amt_in, location_in, payment_in],
        outputs=[status, plot_output, data_table]
    )
    refresh_chart_btn.click(fn=get_pie_chart, outputs=plot_output)
    refresh_table_btn.click(fn=get_raw_data, outputs=data_table)

    # 啟動時預載
    demo.load(fn=get_pie_chart, outputs=plot_output)
    demo.load(fn=get_raw_data, outputs=data_table)

demo.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://380b4f4a8b7c1d910a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:81: UserWarning: Glyph 25903 (\N{CJK UNIFIED IDEOGRAPH-652F}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:81: UserWarning: Glyph 20986 (\N{CJK UNIFIED IDEOGRAPH-51FA}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:81: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:81: UserWarning: Glyph 39006 (\N{CJK UNIFIED IDEOGRAPH-985E}) missing from font(s) DejaVu Sans.
  plt.savefig(output_bytes, format=fmt)
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:81: UserWarning: Glyph 20308 (\N{CJK UNIFIED IDEOGRAPH-4F54}) missing from font(s) DejaVu Sans.
  plt.savefig(output_byte

Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 

/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 39135 (\N{CJK UNIFIED IDEOGRAPH-98DF}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 29289 (\N{CJK UNIFIED IDEOGRAPH-7269}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 39154 (\N{CJK UNIFIED IDEOGRAPH-98F2}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 26009 (\N{CJK UNIFIED IDEOGRAPH-6599}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 25903 (\N{CJK UNIFIED IDEOGRAPH-652F}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph